In [ ]:
# ═══════════════════════════════════════════════════════════════
# STABLE MONEY AI AVATAR — SINGLE-CELL COLAB SETUP
# ═══════════════════════════════════════════════════════════════
# This cell sets up everything: deps, Wav2Lip, server, ngrok.
# Just press Play — all keys are pre-configured.
# Requirements: Colab with T4 GPU (Runtime > Change runtime > T4)
# ═══════════════════════════════════════════════════════════════

import os, sys, time, threading, base64

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CONFIG — API KEYS (pre-configured)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GROQ_API_KEY       = "YOUR_GROQ_API_KEY"
NGROK_TOKEN        = "3BO5O8HXbvLr8j8TjHYPj5UFm4r_3BYg4BqamLqx7t7et34y9"
ELEVENLABS_API_KEY = "sk_a4937e5666c911b91020b9fd09abbd0df3e7bede7d7e958a"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["ELEVENLABS_API_KEY"] = ELEVENLABS_API_KEY

# ── STEP 1: System dependencies ──
print("📦 Installing system dependencies...")
os.system("apt-get update -qq && apt-get install -y -qq ffmpeg espeak-ng > /dev/null 2>&1")
print("✅ System deps installed")

# ── STEP 2: Python packages ──
print("📦 Installing Python packages...")
os.system("pip install -q fastapi 'uvicorn[standard]' python-multipart aiofiles edge-tts groq pyngrok nest_asyncio opencv-python-headless librosa face_alignment elevenlabs")
print("✅ Python packages installed")

# ── STEP 3: GPU check ──
import torch
HAS_GPU = torch.cuda.is_available()
print(f"🖥️  GPU: {torch.cuda.get_device_name(0) if HAS_GPU else 'CPU only (no lip-sync)'}"  )

# ── STEP 4: Wav2Lip setup (GPU only) ──
if HAS_GPU:
    print("📦 Setting up Wav2Lip...")
    if not os.path.exists("/content/Wav2Lip"):
        os.system("git clone -q https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip")
    os.makedirs("/content/Wav2Lip/checkpoints", exist_ok=True)
    CKPT = "/content/Wav2Lip/checkpoints/wav2lip_gan.pth"
    if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 100_000_000:
        print("⬇️  Downloading Wav2Lip checkpoint from HuggingFace...")
        os.system(f'wget -q --show-progress -O {CKPT} "https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth"')
    ckpt_size = os.path.getsize(CKPT) / 1e6 if os.path.exists(CKPT) else 0
    if ckpt_size < 100:
        print(f"⚠️  Checkpoint too small ({ckpt_size:.1f} MB) — trying fallback...")
        os.system(f'wget -q --show-progress -O {CKPT} "https://huggingface.co/numz/wav2lip_studio/resolve/main/checkpoints/wav2lip_gan.pth"')
        ckpt_size = os.path.getsize(CKPT) / 1e6 if os.path.exists(CKPT) else 0
    os.makedirs("/content/checkpoints", exist_ok=True)
    if not os.path.exists("/content/checkpoints/wav2lip_gan.pth"):
        os.symlink(CKPT, "/content/checkpoints/wav2lip_gan.pth")
    print(f"✅ Wav2Lip checkpoint: {ckpt_size:.1f} MB")
else:
    print("⚠️  No GPU — running in audio-only mode (no lip-sync)")

# ── STEP 5: Write server.py + knowledge_base.txt (inline, no CDN) ──
print("📝 Writing server files...")
SERVER_B64 = "..."
KB_B64 = "PT09IFNUQUJMRSBNT05FWSDigJQgT0ZGSUNJQUwgS05PV0xFREdFIEJBU0UgPT09Ckxhc3QgdXBkYXRlZDogTWFyY2ggMjAyNgoKQ09NUEFOWSBPVkVSVklFVwotIFN0YWJsZSBNb25leSBpcyBJbmRpYSdzIGxlYWRpbmcgZml4ZWQtaW5jb21lIGludmVzdG1lbnQgcGxhdGZvcm0KLSBIZWFkcXVhcnRlcmVkIGluIEJhbmdhbG9yZSwgSW5kaWEKLSBSZWd1bGF0ZWQgaW52ZXN0bWVudHMgb25seSDigJQgYWxsIHBhcnRuZXIgaW5zdGl0dXRpb25zIGFyZSBSQkktcmVndWxhdGVkCgpQUk9EVUNUUwoxLiBGaXhlZCBEZXBvc2l0cyAoRkRzKQogICAtIFBhcnRuZXIgaW5zdGl0dXRpb25zOiAyNSsgTkJGQ3MgKE5vbi1CYW5raW5nIEZpbmFuY2lhbCBDb21wYW5pZXMpIGFuZCBTbWFsbCBGaW5hbmNlIEJhbmtzCiAgIC0gUmV0dXJuczogOCUgdG8gOS41JSBwZXIgYW5udW0gKHZzIDYuNeKAkzclIGZyb20gcmVndWxhciBiYW5rcykKICAgLSBUZW51cmVzOiA2IG1vbnRocyB0byA1IHllYXJzCiAgIC0gTWluaW11bSBpbnZlc3RtZW50OiBScyAxLDAwMAogICAtIERJQ0dDLWluc3VyZWQgdXAgdG8gUnMgNSBsYWtocyBwZXIgZGVwb3NpdG9yIHBlciBiYW5rCiAgIC0gU3VpdGFibGUgZm9yOiBjb25zZXJ2YXRpdmUgaW52ZXN0b3JzIHNlZWtpbmcgYmV0dGVyLXRoYW4tYmFuayByZXR1cm5zCgoyLiBTdWtvb24KICAgLSBObyBsb2NrLWluIHBlcmlvZCDigJQgd2l0aGRyYXcgYW55dGltZQogICAtIFJldHVybnM6IDclIHRvIDglIHBlciBhbm51bQogICAtIElkZWFsIGZvcjogaWRsZS9lbWVyZ2VuY3kgbW9uZXksIHNob3J0LXRlcm0gcGFya2luZwogICAtIE1pbmltdW0gaW52ZXN0bWVudDogUnMgMSwwMDAKClNBRkVUWSAmIFJFR1VMQVRJT04KLSBBbGwgcGFydG5lciBOQkZDcyBhbmQgU0ZCcyBhcmUgUkJJLXJlZ3VsYXRlZAotIERJQ0dDIChEZXBvc2l0IEluc3VyYW5jZSBhbmQgQ3JlZGl0IEd1YXJhbnRlZSBDb3Jwb3JhdGlvbikgaW5zdXJlcyBkZXBvc2l0cyB1cCB0byBScyA1IGxha2hzCi0gU3RhYmxlIE1vbmV5IGlzIG5vdCBhIGJhbmsg4oCUIGl0IGlzIGEgbWFya2V0cGxhY2UvcGxhdGZvcm0KLSBJbnZlc3RtZW50cyBhcmUgaGVsZCBkaXJlY3RseSB3aXRoIHBhcnRuZXIgaW5zdGl0dXRpb25zLCBub3Qgd2l0aCBTdGFibGUgTW9uZXkgaXRzZWxmCgpDT01QQVJJU09OIFdJVEggQkFOSyBGRHMKLSBSZWd1bGFyIGJhbmsgRkRzOiB+Ni41JSB0byA3JSByZXR1cm5zCi0gU3RhYmxlIE1vbmV5IEZEczogOCUgdG8gOS41JSByZXR1cm5zCi0gU2FtZSBESUNHQyBpbnN1cmFuY2UgcHJvdGVjdGlvbgotIFdpZGVyIGNob2ljZSBvZiBpbnN0aXR1dGlvbnMKCldITyBTSE9VTEQgSU5WRVNUCi0gSW5kaXZpZHVhbHMgc2Vla2luZyBzYWZlLCBwcmVkaWN0YWJsZSByZXR1cm5zCi0gQW55b25lIHdpdGggaWRsZSBzYXZpbmdzIGVhcm5pbmcgbGVzcyB0aGFuIDclCi0gUmV0aXJlZXMgb3IgY29uc2VydmF0aXZlIGludmVzdG9ycwotIFBlb3BsZSBidWlsZGluZyBhbiBlbWVyZ2VuY3kgZnVuZCAoU3Vrb29uIHByb2R1Y3QpCgpIT1cgVE8gR0VUIFNUQVJURUQKLSBWaXNpdCBzdGFibGVtb25leS5pbgotIENvbXBsZXRlIEtZQyAoUEFOICsgQWFkaGFhcikKLSBDaG9vc2UgRkQgb3IgU3Vrb29uCi0gSW52ZXN0IGZyb20gUnMgMSwwMDAKCklNUE9SVEFOVCBESVNDTEFJTUVSUwotIFBhc3QgcmV0dXJucyBhcmUgaW5kaWNhdGl2ZSwgbm90IGd1YXJhbnRlZWQKLSBJbnZlc3RtZW50cyBhYm92ZSBScyA1IGxha2hzIGFyZSBub3QgRElDR0MtaW5zdXJlZCBiZXlvbmQgdGhlIGxpbWl0Ci0gVGhpcyBpcyBub3QgYSBzYXZpbmdzIGFjY291bnQg4oCUIGl0IGlzIGEgdGVybSBkZXBvc2l0Cg=="

with open("/content/server.py", "wb") as f:
    f.write(base64.b64decode(SERVER_B64))
with open("/content/knowledge_base.txt", "wb") as f:
    f.write(base64.b64decode(KB_B64))
print(f"✅ server.py: {os.path.getsize('/content/server.py')} bytes")
print(f"✅ knowledge_base.txt: {os.path.getsize('/content/knowledge_base.txt')} bytes")

# ── STEP 6: Download frontend + avatar from GitHub ──
print("📥 Downloading frontend...")
REPO = "https://raw.githubusercontent.com/aayushjha-blip/stable-money-avatar/main"
os.makedirs("/content/static", exist_ok=True)
os.system(f"curl -sL {REPO}/static/index.html -o /content/static/index.html")
os.system(f"curl -sL {REPO}/static/mugshot.jpg -o /content/static/mugshot.jpg")
if not os.path.exists("/content/avatar.jpg") and os.path.exists("/content/static/mugshot.jpg"):
    os.system("cp /content/static/mugshot.jpg /content/avatar.jpg")
print(f"✅ index.html: {os.path.getsize('/content/static/index.html')} bytes")

# ── STEP 7: Start ngrok + server ──
from pyngrok import conf, ngrok
conf.get_default().auth_token = NGROK_TOKEN

import nest_asyncio, uvicorn
nest_asyncio.apply()

os.chdir("/content")
sys.path.insert(0, "/content")
if HAS_GPU:
    sys.path.insert(0, "/content/Wav2Lip")

# Kill old server + tunnel
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
time.sleep(2)
ngrok.kill()
time.sleep(1)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print("=" * 55)
print("🌐  PUBLIC URL:", public_url)
print("🔌  WS URL:", public_url.replace("https://", "wss://") + "/ws/talk")
print("🔊  TTS: ElevenLabs")
print("=" * 55)

from server import app as fastapi_app

config = uvicorn.Config(fastapi_app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

print("✅ Server running! Open the PUBLIC URL above in your browser.")
time.sleep(3)

# Keep cell alive
while True:
    time.sleep(60)
